# BioMed RAG — End-to-End Demo
**Stack:** LightRAG + BioMistral-7B-AWQ (vLLM) + nomic-embed-text-v1.5 (vLLM)

### Prerequisites
1. Start the vLLM + LightRAG servers:
   ```bash
   cd biomed-rag && bash start_lightrag_vllm.sh
   ```
2. Wait for the "Services are up" message before running this notebook.

---

| Step | What happens |
|------|-------------|
| 1 | Environment setup / imports |
| 2 | Initialise LightRAG pipeline |
| 3 | Ingest BC5CDR training corpus (one-time) |
| 4 | Interactive queries |
| 5 | CID Relation F1 evaluation |
| 6 | RAGAS quality metrics |


## 1. Environment Setup

In [1]:
import sys, os, logging

# Project root → biomed-rag/
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("biomed_demo")

print("Project root:", project_root)
print("Python:", sys.version)


Project root: /home/hngoc/nlp/biomed-rag
Python: 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]


In [2]:
# Quick connectivity check — both vLLM servers must respond before proceeding
import urllib.request, json

def check_server(url, name):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            data = json.loads(r.read())
        models = [m["id"] for m in data.get("data", [])]
        print(f"✓ {name} — models: {models}")
        return True
    except Exception as e:
        print(f"✗ {name} — NOT REACHABLE: {e}")
        return False

llm_ok   = check_server("http://localhost:8080/v1/models", "LLM server  (port 8080)")
embed_ok = check_server("http://localhost:8081/v1/models", "Embed server (port 8081)")

if not (llm_ok and embed_ok):
    print("\n⚠  Start the servers first:  bash start_lightrag_vllm.sh")


✗ LLM server  (port 8080) — NOT REACHABLE: <urlopen error [Errno 111] Connection refused>
✓ Embed server (port 8081) — models: ['nomic-ai/nomic-embed-text-v1.5']

⚠  Start the servers first:  bash start_lightrag_vllm.sh


## 2. Initialise LightRAG Pipeline

In [ ]:
from module.RAG_pipeline.pipeline.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(mode="hybrid")
await pipeline.initialize()
print("LightRAG ready ✓")


## 3. Ingest BC5CDR Corpus
Run **once** to build the knowledge graph + vector index.  
Subsequent runs will skip already-indexed documents automatically.

In [ ]:
# Ingest BC5CDR Training split (~1 500 PubMed abstracts)
# This takes several minutes the first time — the KG is cached for future runs.
await pipeline.ingest_bc5cdr(split="Training", batch_size=10)
print("Ingestion complete ✓")


## 4. Interactive Queries
Try `local`, `global`, and `hybrid` retrieval modes to see the difference.

In [ ]:
QUESTION = "What diseases are caused by aspirin use?"

for mode in ["local", "global", "hybrid"]:
    print(f"\n{'='*60}")
    print(f"Mode: {mode}")
    print('='*60)
    answer = await pipeline.query(QUESTION, mode=mode)
    print(answer)


## 5. CID Relation Extraction — F1 Evaluation
Measures how well the system identifies Chemical–Disease Induction (CID) relations
against the BC5CDR gold standard.

In [ ]:
from module.RAG_pipeline.evaluate import evaluate_cid_f1
import pandas as pd

# Use max_pairs=50 for a quick smoke-test; set to None for the full Test split
cid_results = await evaluate_cid_f1(pipeline, split="Test", max_pairs=50)

pd.DataFrame([cid_results]).T.rename(columns={0: "value"})


## 6. RAGAS Quality Metrics
Measures faithfulness, answer relevance, context precision/recall.  
Provide your own `qa_pairs` list — each item needs `question` and `ground_truth`.

In [ ]:
from module.RAG_pipeline.evaluate import evaluate_ragas

# Sample QA pairs — replace with actual BioASQ / PubMedQA test items
sample_qa = [
    {
        "question": "What are the main adverse effects of methotrexate?",
        "ground_truth": "Methotrexate can cause hepatotoxicity, mucositis, myelosuppression, and pulmonary toxicity.",
    },
    {
        "question": "Does cisplatin cause nephrotoxicity?",
        "ground_truth": "Yes, cisplatin is associated with dose-dependent nephrotoxicity.",
    },
]

ragas_scores = await evaluate_ragas(pipeline, sample_qa)
pd.DataFrame([ragas_scores]).T.rename(columns={0: "score"})


In [ ]:
# Cleanup — flush LightRAG caches / close DB connections
await pipeline.close()
print("Pipeline closed ✓")
